In [1]:
import os
import sys
import requests
from openai import OpenAI
import csv
import time
from pathlib import Path
from dotenv import load_dotenv
# Add parent directory to sys.path
parent_dir = os.path.abspath("..")
sys.path.append(parent_dir)
from backend.constants import (
    ANKI_CONNECT_URL
)
env_path = Path.cwd().parent / "backend" / ".env"
print(env_path)
load_dotenv(dotenv_path=env_path)

/Users/sethdonaldson/sourcecode/anki-vocab-generator/backend/.env


True

In [2]:


# ---------------------- Configuration ----------------------
DECK_NAME = "8000+ most common swedish words"
MODEL_NAME = "Memrise - 8000+ Most Common Swedish Words - Part 1 (of four) - Swedish"
VOCAB_FIELD = "Swedish"
DEFINITION_FIELD = "English"
EX_SENTENCE_SW_FIELD = "Example Sentence (Swedish)"
EX_SENTENCE_EN_FIELD = "Example Sentence (English)"
EX_SENTECE_AUDIO_FIELD = "Example Sentence Audio"

# CSV output file
CSV_OUTPUT_FILE = "swedish_examples.csv"

# ---------------------- AnkiConnect Helper Functions ----------------------
def request(action, **params):
    return {"action": action, "params": params, "version": 6}

def invoke(action, **params):
    payload = request(action, **params)
    response = requests.post(ANKI_CONNECT_URL, json=payload)
    if not response.ok:
        raise Exception(f"Request failed with status code {response.status_code}: {response.text}")
    response_json = response.json()
    if 'error' not in response_json or 'result' not in response_json:
        raise Exception('Invalid response structure')
    if response_json['error'] is not None:
        raise Exception(response_json['error'])
    return response_json['result']

# ---------------------- LLM Generation Function ----------------------
def generate_examples(vocab_word, definition):
    """
    Using the vocabulary word (in Swedish) as context,
    generate a Swedish example sentence and its English translation.
    
    Returns a tuple: (swedish_sentence, english_sentence)
    """
    prompt = (f"""Generate a Swedish example sentence that naturally uses the word '{vocab_word}' with the given definition: '{definition}'.
              
              Keep it simple, simple, and memorable. The purpose is to help the user remember the word, as the sentence 
              (and its english translation) will be used in an Anki flashcard for learning the vocabulary word.
              I am a beginner trying to learn Swedish, so make sure the sentence is simple and easy to understand.
              Don't be afraid to use common Swedish phrases and idioms (as appropriate). The purpose is to help the user learn common, spoken/written Swedish.
              
              THEN provide the English translation for that sentence on a new line.
              Return the answer in exactly two lines: first the Swedish sentence, then the English sentence. With no other text.
              
              IMPORTANT: if the vocabulary word is a verb, it will appear as "att" + the verb. This doesn't necessarily mean you should use "att" in your example sentence.
              Just use the verb as you would in a normal sentence.
              IMPORTANT: if the vocabulary word is a noun, it will appear as "en" or "ett" + the noun. This doesn't necessarily mean you should use "en" or "ett" in your example sentence.
              Just use the noun as you would in a normal sentence.
              
              EXAMPLE: If the vocab word = vatten
              You would output something like this:
              Vatten är en vätska
              Water is a liquid
              """)
    try:
        client = OpenAI(api_key=os.getenv("OPENAI_API_KEY")) 
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "You generate natural-sounding Swedish sentences and their English translations."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.7,
            max_tokens=60,
        )
        
        # Split output into two lines (we expect exactly 2 lines).
        output = response.choices[0].message.content.strip().split("\n")
        if len(output) < 2:
            raise Exception("Unexpected output format from LLM")
        swedish_sentence = output[0].strip()
        english_sentence = output[1].strip()
        return swedish_sentence, english_sentence
    except Exception as e:
        print(f"Error generating example for '{vocab_word}': {e}")
        return None, None


In [3]:
print(OPENAI_API_KEY)

NameError: name 'OPENAI_API_KEY' is not defined

In [4]:
# ---------------------- Main Process ----------------------
def write_examples():
    # Step 1: Get all note IDs for the specified deck.
    query = f'deck:"{DECK_NAME}"'
    note_ids = invoke("findNotes", query=query)
    if not note_ids:
        print(f"No notes found in deck {DECK_NAME}")
        return
    
    # Get note info including fields.
    notes_info = invoke("notesInfo", notes=note_ids)
    
    # Prepare a list to optionally store results in CSV.
    csv_rows = [["Vocabulary Word", "Example Sentence (Swedish)", "Example Sentence (English)"]]
    
    total_notes = len(notes_info)
    print(f"Processing {total_notes} notes...")
    
    for i, note in enumerate(notes_info, start=1):
        note_id = note["noteId"] if "noteId" in note else note["id"]
        fields = note["fields"]
        vocab_word = fields.get(VOCAB_FIELD, {}).get("value", "").strip()
        definition = fields.get(DEFINITION_FIELD, {}).get("value", "").strip()
        if not vocab_word:
            print(f"Note ID {note_id} does not have a value for '{VOCAB_FIELD}'. Skipping.")
            continue
        
        # swedish_sentence = fields.get(EX_SENTENCE_SW_FIELD, {}).get("value", "").strip()
        # if swedish_sentence:
        #     print(f"[{i}/{total_notes}] Skipping note ID {note_id} because it already has an example sentence.")
        #     continue
        
        
        print(f"[{i}/{total_notes}] Generating examples for '{vocab_word}'")
        swedish_sentence, english_sentence = generate_examples(vocab_word, definition)
        
        if not swedish_sentence or not english_sentence:
            print(f"Skipping note ID {note_id} due to generation error.")
            continue
        
        # Update CSV row list.
        csv_rows.append([vocab_word, swedish_sentence, english_sentence])
        
        # Update the note's fields with the generated content.
        update_payload = {
            "id": note_id,
            "fields": {
                EX_SENTENCE_SW_FIELD: swedish_sentence,
                EX_SENTENCE_EN_FIELD: english_sentence
            }
        }
        print(update_payload)
        
        try:
            invoke("updateNoteFields", note=update_payload)
        except Exception as e:
            print(f"Error updating note ID {note_id}: {e}")
        
        # Optional: add a short delay to avoid overloading the API (adjust as needed).
        # time.sleep(1)
    
    # Write CSV file (optional)
    with open(CSV_OUTPUT_FILE, "w", newline="", encoding="utf-8") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerows(csv_rows)
    print(f"CSV file saved to {CSV_OUTPUT_FILE}")

In [5]:
# Equivalent to Python's store_audio_file function, but modified to work with base64 data directly
def storeAudioFile(filename, audioData):
    response = invoke('storeMediaFile',
        filename=filename,
        data=audioData
    )
    print(f"Stored {filename}:", response)
    return response


In [17]:
import io
import base64
from gtts import gTTS
# Function to generate TTS audio
import random
import string


def generate_audio_openai(text, vocab_word, language="swedish"):
    """
    Generate TTS audio for a given text and return base64-encoded audio data
    """
    # generate unique filename (to avoid collisions for same vocab word)
    file_id = ''.join(random.choices(string.ascii_letters + string.digits, k=5))
    filename = f"{''.join(vocab_word.split())}_{file_id}.mp3"
    filepath = f"{language}_audio/{filename}"
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY")) 
    with client.audio.speech.with_streaming_response.create(
        model="gpt-4o-mini-tts",
        voice="ash",
        input=text,
        instructions=f"""You are a native {language} speaker. Speak in a natural, conversational tone.
        You are recording this sentence for an Anki flashcard in a deck of vocabulary flashcards.
        The purpose of the flashcard is to help the user learn the vocab word '{vocab_word}'.
        The text is a sentence that uses the vocab word '{vocab_word}'.
        Focus on speaking naturally. Be engaging. Speak at a natural pace.
        """
    ) as response:
        response.stream_to_file(filepath)
        return filename


In [23]:
# ---------------------- Main Process ----------------------
from backend.pipeline import generate_audio

def write_audios():
    # Step 1: Get all note IDs for the specified deck.
    query = f'deck:"{DECK_NAME}"'
    note_ids = invoke("findNotes", query=query)
    if not note_ids:
        print(f"No notes found in deck {DECK_NAME}")
        return
    
    # Get note info including fields.
    notes_info = invoke("notesInfo", notes=note_ids)
    
    total_notes = len(notes_info)
    print(f"Processing {total_notes} notes...")
    
    for i, note in enumerate(notes_info, start=1):
        note_id = note["noteId"] if "noteId" in note else note["id"]
        fields = note["fields"]
        vocab_word = fields.get(VOCAB_FIELD, {}).get("value", "").strip()
        swedish_sentence = fields.get(EX_SENTENCE_SW_FIELD, {}).get("value", "").strip()
        if not swedish_sentence:
            print(f"Note ID {note_id} does not have a value for '{EX_SENTENCE_SW_FIELD}'. Skipping.")
            continue
        
        sentence_audio = fields.get(EX_SENTECE_AUDIO_FIELD, {}).get("value", "").strip()
        # if sentence_audio:
        #     print(f"[{i}/{total_notes}] Skipping note ID {note_id} because it already has an example audio.")
        #     continue
        
        
        print(f"[{i}/{total_notes}] Generating audio for '{vocab_word}'")
        exampleAudioFilename = generate_audio_openai(swedish_sentence, vocab_word, "swedish")
        
        if not sentence_audio:
            print(f"Skipping note ID {note_id} due to generation error.")
            continue
        
        # Read the MP3 file and convert to base64
        with open(f"swedish_audio/{exampleAudioFilename}", "rb") as audio_file:
            audio_data = audio_file.read()
            base64_audio = base64.b64encode(audio_data).decode('utf-8')

        storeAudioFile(exampleAudioFilename, base64_audio)

        # Update the note's fields with the generated content.
        update_payload = {
            "id": note_id,
            "fields": {
                EX_SENTECE_AUDIO_FIELD: f"[sound:{exampleAudioFilename}]",
            }
        }
        print(update_payload)
        
        try:
            invoke("updateNoteFields", note=update_payload)
        except Exception as e:
            print(f"Error updating note ID {note_id}: {e}")
        
        # break
        
        # Optional: add a short delay to avoid overloading the API (adjust as needed).
        # time.sleep(1)
    

In [24]:
write_audios()

Processing 8327 notes...
[1/8327] Generating audio for 'och'
Stored och_1rXKl.mp3: och_1rXKl.mp3
{'id': 1598022787258, 'fields': {'Example Sentence Audio': '[sound:och_1rXKl.mp3]'}}
[2/8327] Generating audio for 'att vara'
Stored attvara_rLjrW.mp3: attvara_rLjrW.mp3
{'id': 1598022787263, 'fields': {'Example Sentence Audio': '[sound:attvara_rLjrW.mp3]'}}
[3/8327] Generating audio for 'i'
Stored i_EK3yf.mp3: i_EK3yf.mp3
{'id': 1598022787266, 'fields': {'Example Sentence Audio': '[sound:i_EK3yf.mp3]'}}
[4/8327] Generating audio for 'att ha'
Stored attha_kveyS.mp3: attha_kveyS.mp3
{'id': 1598022787269, 'fields': {'Example Sentence Audio': '[sound:attha_kveyS.mp3]'}}
[5/8327] Generating audio for 'dess'
Stored dess_sAVHw.mp3: dess_sAVHw.mp3
{'id': 1598022787273, 'fields': {'Example Sentence Audio': '[sound:dess_sAVHw.mp3]'}}
[6/8327] Generating audio for 'det'
Stored det_HsSvG.mp3: det_HsSvG.mp3
{'id': 1598022787277, 'fields': {'Example Sentence Audio': '[sound:det_HsSvG.mp3]'}}
[7/8327] Ge

FileNotFoundError: [Errno 2] No such file or directory: 'swedish_audio/attloggain/ut_oBXTP.mp3'

In [67]:
main()

Processing 8327 notes...
[1/8327] Generating examples for 'och'
{'id': 1598022787258, 'fields': {'Example Sentence (Swedish)': 'Jag gillar kaffe och te.', 'Example Sentence (English)': 'I like coffee and tea.'}}
[2/8327] Generating examples for 'att vara'
{'id': 1598022787263, 'fields': {'Example Sentence (Swedish)': 'Att vara snäll är viktigt.', 'Example Sentence (English)': 'To be kind is important.'}}
[3/8327] Generating examples for 'i'
{'id': 1598022787266, 'fields': {'Example Sentence (Swedish)': 'Jag bor i en liten stad.', 'Example Sentence (English)': 'I live in a small town.'}}
[4/8327] Generating examples for 'att ha'
{'id': 1598022787269, 'fields': {'Example Sentence (Swedish)': 'Jag har en katt.', 'Example Sentence (English)': 'I have a cat.'}}
[5/8327] Generating examples for 'dess'
{'id': 1598022787273, 'fields': {'Example Sentence (Swedish)': 'Hunden viftade med sin svans, men katten vände sig bort och ignorerade dess lekfullhet.', 'Example Sentence (English)': 'The dog 

In [92]:
openai.Model.list()

APIRemovedInV1: 

You tried to access openai.Model, but this is no longer supported in openai>=1.0.0 - see the README at https://github.com/openai/openai-python for the API.

You can run `openai migrate` to automatically upgrade your codebase to use the 1.0.0 interface. 

Alternatively, you can pin your installation to the old version, e.g. `pip install openai==0.28`

A detailed migration guide is available here: https://github.com/openai/openai-python/discussions/742


In [94]:
!pip show openai

326406.71s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


Name: openai
Version: 1.61.1
Summary: The official Python library for the openai API
Home-page: https://github.com/openai/openai-python
Author: 
Author-email: OpenAI <support@openai.com>
License-Expression: Apache-2.0
Location: /Users/sethdonaldson/sourcecode/anki-vocab-generator/.venv/lib/python3.12/site-packages
Requires: anyio, distro, httpx, jiter, pydantic, sniffio, tqdm, typing-extensions
Required-by: 
